# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(metadata['name'])
print('Description:', metadata['description'])
print('Published:', metadata['datePublished'])
print('Keywords:', metadata.get('keywords', []))


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema defines multiple entities. We will enumerate available record sets, fields, and columns—all uniquely referenced by their `@id`.

Let's print an overview of available record sets and their fields.

In [ ]:
def get_record_sets(ds):
    record_sets = ds.metadata.to_json().get('recordSet', [])
    if isinstance(record_sets, dict):
        return [record_sets]
    return record_sets

# Get record sets
record_sets = get_record_sets(dataset)
if not record_sets:
    print('No record sets found in metadata.')
else:
    print('Available record sets:')
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name','')} | description: {rs.get('description','')}")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            print(f"    Field @id: {field['@id']} | name: {field.get('name','')} | description: {field.get('description','')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

All entities are referenced by their `@id`. We use `record_set` IDs to load records.

In [ ]:
# Extract record set @ids
record_sets = get_record_sets(dataset)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded records for {record_set_id}:")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records for {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, grouping by categorical attributes. All fields are referenced by their `@id`.

Let's pick the first loaded record set for demonstration.

In [ ]:
import numpy as np

# Pick the first record set with records
chosen_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        chosen_rs_id = rs_id
        break

if chosen_rs_id is not None:
    print(f"Analyzing record set: {chosen_rs_id}")
    df = dataframes[chosen_rs_id]
    print('Columns:', df.columns.tolist())

    # Identify a numeric column (for demo, pick first float/int column)
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break

    if numeric_field_id:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field if available
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in records.")
else:
    print("No record set with loaded records for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot histograms and barplots for the numeric and grouping fields determined above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(7,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading metadata and tabular data from a Croissant schema dataset using `mlcroissant`.
- Each dataset entity was referenced by its unique `@id`, in accordance with FAIR principles.
- We performed basic data extraction, normalization, filtering, and visualization for exploratory analysis.
- The FAIR^2 dataset provides rich clinicopathological and molecular data for cancer survivors with second primary colorectal cancer, useful for biomarker analysis and clinical research.


<br>
*Notebook generated using mlcroissant and FAIR^2 schema. For more documentation, see [mlcroissant docs](https://mlcroissant.org/).*